# Data Cleaning Pipeline

## Install Packages

In [28]:
!pip install contextily

In [29]:
# packages
from google.colab import files
import io
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import seaborn as sns
import contextily as ctx
from sklearn.cluster import KMeans
import matplotlib.patches as mpatches

## Read in Raw Datasets

### Original Policing Dataset

In [30]:
## Read in SOPP Dataset from Website provided
sopp_df = pd.read_csv('nashville.csv')

/tmp/ipykernel_993/4052984372.py:2: DtypeWarning: Columns (6,8,15,16,17,22,23,24,25,29,30,31,32,33,35,36,37,38,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  sopp_df = pd.read_csv('tn_nashville_2020_04_01.csv')


### Original Crash Datasets

In [31]:
## Read in 2009 - 2019 Crash Dataset and Merge Each Year's Crash Dataset from Website provided
# 2009
acc9_df = pd.read_csv('ACCIDENT9.CSV')

acc9_tn_df = acc9_df[acc9_df['STATE'] == 47]
acc9_nash_df = acc9_tn_df[acc9_tn_df['CITY'] == 1760]

# 2010
acc10_df = pd.read_csv('ACCIDENT10.CSV')

acc10_tn_df = acc10_df[acc10_df['STATE'] == 47]
acc10_nash_df = acc10_tn_df[acc10_tn_df['CITY'] == 1760]

# 2011
acc11_df = pd.read_csv('ACCIDENT11.CSV')

acc11_tn_df = acc11_df[acc11_df['STATE'] == 47]
acc11_nash_df = acc11_tn_df[acc11_tn_df['CITY'] == 1760]

# 2012
acc12_df = pd.read_csv('ACCIDENT12.csv')

acc12_tn_df = acc12_df[acc12_df['STATE'] == 47]
acc12_nash_df = acc12_tn_df[acc12_tn_df['CITY'] == 1760]

# 2013
acc13_df = pd.read_csv('ACCIDENT13.csv')

acc13_tn_df = acc13_df[acc13_df['STATE'] == 47]
acc13_nash_df = acc13_tn_df[acc13_tn_df['CITY'] == 1760]

# 2014
acc14_df = pd.read_csv('ACCIDENT14.csv')

acc14_tn_df = acc14_df[acc14_df['STATE'] == 47]
acc14_nash_df = acc14_tn_df[acc14_tn_df['CITY'] == 1760]

# 2015
acc15_df = pd.read_csv('accident15.csv')

acc15_tn_df = acc15_df[acc15_df['STATE'] == 47]
acc15_nash_df = acc15_tn_df[acc15_tn_df['CITY'] == 1760]

# 2016
acc16_df = pd.read_csv('accident16.CSV')

acc16_tn_df = acc16_df[acc16_df['STATE'] == 47]
acc16_nash_df = acc16_tn_df[acc16_tn_df['CITY'] == 1760]

# 2017
acc17_df = pd.read_csv('accident17.CSV')

acc17_tn_df = acc17_df[acc17_df['STATE'] == 47]
acc17_nash_df = acc17_tn_df[acc17_tn_df['CITY'] == 1760]

# 2018
acc18_df = pd.read_csv('accident18.csv')

acc18_tn_df = acc18_df[acc18_df['STATE'] == 47]
acc18_nash_df = acc18_tn_df[acc18_tn_df['CITY'] == 1760]

# 2019
acc19_df = pd.read_csv('accident19.CSV', encoding='latin1')

acc19_tn_df = acc19_df[acc19_df['STATE'] == 47]
acc19_nash_df = acc19_tn_df[acc19_tn_df['CITY'] == 1760]

/tmp/ipykernel_993/651460981.py:51: DtypeWarning: Columns (40,42) have mixed types. Specify dtype option on import or set low_memory=False.
  acc17_df = pd.read_csv('accident17.CSV')
/tmp/ipykernel_993/651460981.py:63: DtypeWarning: Columns (40,42) have mixed types. Specify dtype option on import or set low_memory=False.
  acc19_df = pd.read_csv('accident19.CSV', encoding='latin1')


## Clean Crash Dataset

In [32]:
## Merge Crash Datasets
acc_nash_df = pd.concat([acc9_nash_df, acc10_nash_df, acc11_nash_df, acc12_nash_df,
                               acc13_nash_df, acc14_nash_df, acc15_nash_df, acc16_nash_df,
                               acc17_nash_df, acc18_nash_df, acc19_nash_df], ignore_index=True)

In [33]:
# Create lists of columns names for each year and find the ones that are present for all
list9 = acc9_nash_df.columns.tolist()
list10 = acc10_nash_df.columns.tolist()
list11 = acc11_nash_df.columns.tolist()
list12 = acc12_nash_df.columns.tolist()
list13 = acc13_nash_df.columns.tolist()
list14 = acc14_nash_df.columns.tolist()
list15 = acc15_nash_df.columns.tolist()
list16 = acc16_nash_df.columns.tolist()
list17 = acc17_nash_df.columns.tolist()
list18 = acc18_nash_df.columns.tolist()
list19 = acc19_nash_df.columns.tolist()

commons = set(list9).intersection(list10, list11, list12, list13, list14, list15, list16, list17, list18, list19)
columns = list(commons)

### Final Crash Dataset

In [34]:
# remove any columns that aren't present for every year
crash_df = acc_nash_df[columns]

## Clean Policing Dataset

### Use a Shape File of Nashville Parcels to ensure geographic accuracy of Policing Dataset

In [35]:
# read in shape file to fix sopp data
parcels = gpd.read_file('nashville-tennessee-parcels.shp')

In [36]:
# create sopp data frame in geopandas
sopp = gpd.GeoDataFrame(
    sopp_df,
    geometry=gpd.points_from_xy(sopp_df.lng, sopp_df.lat),
    crs="EPSG:4326")

In [37]:
# ensure crs are the same for both files
parcels = parcels.to_crs(sopp.crs)
print(sopp.crs)
print(parcels.crs)

EPSG:4326
EPSG:4326


In [38]:
# join them
sopp_within_parcels = gpd.sjoin(
    sopp,
    parcels,
    how="inner",
    predicate="within")

In [39]:
# only include columns from sopp data
sopp_df = sopp_within_parcels[sopp.columns]

In [40]:
# reduce datapoints by reasoning of stop
sopp_df = sopp_df[sopp_df["reason_for_stop"].isin([
    'moving traffic violation',
    'seatbelt violation'])]

In [41]:
sopp_df.to_csv('sopp_df.csv')

## K-Means Clustering Assignment for Zoning Assignments

### Adding Respective Source Column for each Dataset

In [42]:
# assign source column for easy separation later
sopp_df['source'] = 'sopp'
crash_df['source'] = 'crash'

# ensure key column names are consistent
crash_df = crash_df.rename(columns={'LATITUDE': 'lat', 'LONGITUD': 'lng'})

/tmp/ipykernel_993/3212834914.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crash_df['source'] = 'crash'


In [43]:
# concatenate both datasets
combined_df = pd.concat([sopp_df, crash_df])

# apply kmeans
X = combined_df[['lat', 'lng']].values

kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10) #

# fit the model to the data and predict the cluster assignments
y_kmeans = kmeans.fit_predict(X)

# ddd the cluster labels back to the Dataframe
combined_df['cluster_label'] = y_kmeans

In [44]:
# separate files
crash_df_3_8 = combined_df[combined_df['source'] == 'crash']
sopp_df_3_8 = combined_df[combined_df['source'] == 'sopp']